In [ ]:
House prices v2 · PY
コピー

import sys
sys.path.append("../utils")

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from scipy.special import boxcox1p
import warnings
warnings.filterwarnings("ignore")


class CFG:
    exp_name = "house_prices_v2"
    seed = 123
    n_splits = 5
    
    # LightGBM parameters
    lgb_num_leaves = 16
    lgb_learning_rate = 0.03
    lgb_n_estimators = 15000
    lgb_min_child_samples = 30
    lgb_subsample = 0.7
    lgb_colsample_bytree = 0.7
    lgb_reg_alpha = 0.5
    lgb_reg_lambda = 0.5
    
    # XGBoost parameters
    xgb_max_depth = 3
    xgb_learning_rate = 0.03
    xgb_n_estimators = 10000
    xgb_min_child_weight = 5
    
    # CatBoost parameters
    cat_iterations = 10000
    cat_learning_rate = 0.03
    cat_depth = 4
    cat_l2_leaf_reg = 5
    
    # Ridge / Lasso
    ridge_alphas = [10, 15, 20]
    lasso_alphas = [0.0005, 0.001, 0.0015]
    
    # Ensemble weights
    w_lgb = 0.35
    w_xgb = 0.22
    w_ridge = 0.18
    w_lasso = 0.15
    w_cat = 0.10
    
    # Seeds for averaging
    seeds_lgb = [123, 456, 789, 42, 2024]
    seeds_xgb = [123, 456, 789]
    seeds_cat = [123, 456, 789]
    
    # Box-Cox
    boxcox_lambda = 0.15
    skew_threshold = 0.5


class Paths:
    p = "./input/"
    train = p + "train.csv"
    test = p + "test.csv"
    sample = p + "sample_submission.csv"
    submission = f"./output/{CFG.exp_name}.csv"


# ===== ユーティリティ関数 =====
def check_df(df, columns=None, show_values_limit=10):
    """
    DataFrameの各列の情報（データ型、NaN数、ユニーク数、ユニーク値）を一覧表示します

    Parameters:
    - df: 対象のDataFrame
    - columns: 対象の列 (list)。Noneのときは全列
    - show_values_limit: ユニーク値を表示する最大数。これを超える列は「多すぎる」と表示されます

    Returns:
    - info_df: 各列の情報をまとめたDataFrame
    """
    if columns is not None:
        df = df[columns]

    type_series = df.dtypes
    nan_count_series = df.isnull().sum()
    nunique_series = df.nunique()

    unique_values = {}
    for col in df.columns:
        if nunique_series[col] <= show_values_limit:
            unique_values[col] = df[col].unique().tolist()
        else:
            unique_values[col] = f"> {show_values_limit} unique values"

    info_df = pd.DataFrame(
        {
            "dtypes": type_series.astype(str),
            "NaN Count": nan_count_series,
            "Nunique": nunique_series,
            "Unique Values": pd.Series(unique_values),
        }
    )

    info_df = info_df.reset_index().rename(columns={"index": "Column"})

    return info_df


# ===== データ読み込み =====
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)
print(f"train shape: {train.shape}, test shape: {test.shape}")


# ===== EDA（check_dfを使用）=====
if False:  # EDA時はTrueに変更
    display(check_df(train))
    display(check_df(test))
    
    # 目的変数の分布
    target_col = "SalePrice"
    
    plt.figure(figsize=(8, 5))
    plt.hist(train[target_col], bins=50)
    plt.title(f"Distribution of {target_col}")
    plt.show()
    
    plt.figure(figsize=(8, 5))
    plt.boxplot(train[target_col])
    plt.title(f"Boxplot of {target_col}")
    plt.grid()
    plt.show()


# ===== 外れ値除去 =====
def remove_outliers(df):
    df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000))]
    return df

train = remove_outliers(train)


# ===== 目的変数 =====
y_train = np.log1p(train["SalePrice"].values)
train_id = train["Id"]
test_id = test["Id"]
print(f"y_train shape: {y_train.shape}")


# ===== 特徴量エンジニアリング =====
def feature_engineering(df):
    df = df.copy()
    
    # 交互作用・合計特徴量
    df["QualGrLiv"] = df["OverallQual"] * df["GrLivArea"]
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["QualTotalSF"] = df["OverallQual"] * df["TotalSF"]
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalBath"] = (df["FullBath"] + 0.5 * df["HalfBath"].fillna(0) +
                       df["BsmtFullBath"].fillna(0) + 0.5 * df["BsmtHalfBath"].fillna(0))
    df["AreaPerRoom"] = df["GrLivArea"] / df["TotRmsAbvGrd"].replace(0, 1)
    df["GarageScore"] = df["GarageCars"] * df["GarageArea"]
    
    # Log変換特徴量
    df["Log_GrLivArea"] = np.log1p(df["GrLivArea"])
    df["Log_LotArea"] = np.log1p(df["LotArea"])
    df["Log_TotalSF"] = np.log1p(df["TotalSF"])
    
    # 追加特徴量
    df["QualGrLivLog"] = df["OverallQual"] * df["Log_GrLivArea"]
    df["TotalPorchSF"] = (df["OpenPorchSF"] + df["EnclosedPorch"] +
                          df["3SsnPorch"].fillna(0) + df["ScreenPorch"].fillna(0))
    df["IsNew"] = (df["Age"] <= 5).astype(int)
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
    df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["BsmtScore"] = df["TotalBsmtSF"] * df["HasBsmt"]
    
    return df


def fill_missing(df, num_cols, cat_cols):
    df = df.copy()
    for col in num_cols:
        df[col] = df[col].fillna(0)
    for col in cat_cols:
        df[col] = df[col].fillna("None")
    return df


def apply_boxcox(train_df, test_df, num_cols):
    train_df = train_df.copy()
    test_df = test_df.copy()
    skewed_features = []
    
    for col in num_cols:
        all_data = pd.concat([train_df[col], test_df[col]])
        skewness = all_data.skew()
        
        if abs(skewness) > CFG.skew_threshold:
            skewed_features.append((col, skewness))
            train_df[col] = boxcox1p(train_df[col], CFG.boxcox_lambda)
            test_df[col] = boxcox1p(test_df[col], CFG.boxcox_lambda)
    
    print(f"✅ Box-Cox: {len(skewed_features)}個の特徴量を変換")
    return train_df, test_df


def encode_onehot(train_df, test_df, cat_cols):
    train_df['is_train'] = 1
    test_df['is_train'] = 0
    combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    
    combined_encoded = pd.get_dummies(
        combined,
        columns=cat_cols,
        drop_first=True,
        dtype=int
    )
    
    train_encoded = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
    test_encoded = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)
    
    return train_encoded, test_encoded


# ===== 前処理パイプライン =====
train_processed = feature_engineering(train)
test_processed = feature_engineering(test)

# 特徴量作成後のEDA
if False:  # EDA時はTrueに変更
    display(check_df(train_processed))
    display(check_df(test_processed))

train_processed = train_processed.drop(columns=["Id", "SalePrice"])
test_processed = test_processed.drop(columns=["Id"])

num_cols = train_processed.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train_processed.select_dtypes(include=["object"]).columns.tolist()
print(f"数値特徴量: {len(num_cols)}個, カテゴリ特徴量: {len(cat_cols)}個")

train_processed = fill_missing(train_processed, num_cols, cat_cols)
test_processed = fill_missing(test_processed, num_cols, cat_cols)

train_processed, test_processed = apply_boxcox(train_processed, test_processed, num_cols)
train_processed, test_processed = encode_onehot(train_processed, test_processed, cat_cols)

X_train = train_processed.values
X_test = test_processed.values
feature_names = train_processed.columns.tolist()

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}")


# ===== Cross Validation (LightGBM) =====
params_lgb = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": CFG.lgb_num_leaves,
    "learning_rate": CFG.lgb_learning_rate,
    "n_estimators": CFG.lgb_n_estimators,
    "min_child_samples": CFG.lgb_min_child_samples,
    "subsample": CFG.lgb_subsample,
    "colsample_bytree": CFG.lgb_colsample_bytree,
    "reg_alpha": CFG.lgb_reg_alpha,
    "reg_lambda": CFG.lgb_reg_lambda,
    "random_state": CFG.seed,
    "verbose": -1,
}

cv = KFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
metrics = []
imp = pd.DataFrame()

print("\n" + "=" * 30 + " Cross Validation " + "=" * 30)
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print("=" * 10, f"Fold {fold + 1}", "=" * 10)
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]
    
    model = lgb.LGBMRegressor(**params_lgb)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(100),
        ],
    )
    
    y_tr_pred = model.predict(X_tr)
    y_val_pred = model.predict(X_val)
    
    rmse_tr = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
    rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
    metrics.append([fold + 1, rmse_tr, rmse_val])
    
    _imp = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_,
        "fold": fold + 1,
    })
    imp = pd.concat([imp, _imp], axis=0, ignore_index=True)

metrics_array = np.array(metrics)
print("\n[cv ] tr: {:.5f} ± {:.5f}, va: {:.5f} ± {:.5f}".format(
    metrics_array[:, 1].mean(), metrics_array[:, 1].std(),
    metrics_array[:, 2].mean(), metrics_array[:, 2].std(),
))


# ===== Feature Importance =====
imp_mean = imp.groupby("feature")["importance"].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x=imp_mean.head(30).values, y=imp_mean.head(30).index)
plt.title("Top 30 Feature Importances (LightGBM)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


# ===== 予測関数 =====
def predict_lgb(X_train, y_train, X_test, params, seeds):
    predictions = []
    for seed in seeds:
        params['random_state'] = seed
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_train,
                  eval_set=[(X_train, y_train)],
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        predictions.append(model.predict(X_test))
    return np.mean(predictions, axis=0)


def predict_xgb(X_train, y_train, X_test, seeds):
    predictions = []
    for seed in seeds:
        model = xgb.XGBRegressor(
            max_depth=CFG.xgb_max_depth,
            learning_rate=CFG.xgb_learning_rate,
            n_estimators=CFG.xgb_n_estimators,
            min_child_weight=CFG.xgb_min_child_weight,
            subsample=CFG.lgb_subsample,
            colsample_bytree=CFG.lgb_colsample_bytree,
            reg_alpha=CFG.lgb_reg_alpha,
            reg_lambda=CFG.lgb_reg_lambda,
            random_state=seed,
            early_stopping_rounds=100,
        )
        model.fit(X_train, y_train, eval_set=[(X_train, y_train)], verbose=False)
        predictions.append(model.predict(X_test))
    return np.mean(predictions, axis=0)


def predict_catboost(X_train, y_train, X_test, seeds):
    predictions = []
    for seed in seeds:
        model = CatBoostRegressor(
            iterations=CFG.cat_iterations,
            learning_rate=CFG.cat_learning_rate,
            depth=CFG.cat_depth,
            l2_leaf_reg=CFG.cat_l2_leaf_reg,
            random_seed=seed,
            verbose=False,
            early_stopping_rounds=100,
        )
        model.fit(X_train, y_train, eval_set=(X_train, y_train), verbose=False)
        predictions.append(model.predict(X_test))
    return np.mean(predictions, axis=0)


def predict_ridge(X_train, y_train, X_test, alphas):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    predictions = []
    for alpha in alphas:
        model = Ridge(alpha=alpha)
        model.fit(X_train_scaled, y_train)
        predictions.append(model.predict(X_test_scaled))
    return np.mean(predictions, axis=0)


def predict_lasso(X_train, y_train, X_test, alphas):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    predictions = []
    for alpha in alphas:
        model = Lasso(alpha=alpha, max_iter=10000)
        model.fit(X_train_scaled, y_train)
        predictions.append(model.predict(X_test_scaled))
    return np.mean(predictions, axis=0)


# ===== 各モデルで予測 =====
print("\n" + "=" * 30 + " Prediction " + "=" * 30)

print("LightGBM...")
pred_lgb = predict_lgb(X_train, y_train, X_test, params_lgb, CFG.seeds_lgb)

print("XGBoost...")
pred_xgb = predict_xgb(X_train, y_train, X_test, CFG.seeds_xgb)

print("CatBoost...")
pred_cat = predict_catboost(X_train, y_train, X_test, CFG.seeds_cat)

print("Ridge...")
pred_ridge = predict_ridge(X_train, y_train, X_test, CFG.ridge_alphas)

print("Lasso...")
pred_lasso = predict_lasso(X_train, y_train, X_test, CFG.lasso_alphas)

print("✅ 全モデル予測完了")


# ===== アンサンブル =====
y_pred_log = (
    CFG.w_lgb * pred_lgb +
    CFG.w_xgb * pred_xgb +
    CFG.w_ridge * pred_ridge +
    CFG.w_lasso * pred_lasso +
    CFG.w_cat * pred_cat
)
y_pred = np.expm1(y_pred_log)

print(f"\n平均予測価格: ${y_pred.mean():,.0f}")


# ===== 提出ファイル =====
df_submit = pd.DataFrame({
    "Id": test_id,
    "SalePrice": y_pred,
})
df_submit.to_csv(Paths.submission, index=False)
print(f"✅ 保存完了: {Paths.submission}")